# Diagnostic: why did variant A's W1 time change?

The same W1 pipeline measured 7.13s in an earlier benchmark run and 45.97s in
the current one — 6.4x slower with no change to the pipeline itself.

**Run this in a fresh kernel, before anything else touches the data.** No Spark
session is created here, deliberately: the point is to measure MongoDB with
nothing else competing for the page cache.

Three candidate causes, and what distinguishes them:

| Cause | Signature |
|---|---|
| Page-cache contention from the Spark reads | A is fast here in isolation |
| `allowDiskUse=True`, added between runs | The two timings below differ |
| Collection growth or a lost index | Counts differ from the snapshot, or the plan shows a collection scan |

One cause is already ruled out by the benchmark itself: it is not a cold MongoDB
cache. W3 scans all 1,171,373 company documents in 0.44s in the same session,
which is not achievable from disk.

This notebook establishes which of the remaining three it is. It does not assume
an answer.

In [1]:
import json
import statistics
import time

from pymongo import MongoClient

MONGO_URI = "mongodb://mongodb:27017"
MONGO_DB = "companiesdb"

# Snapshot 2026-08-25, the counts the benchmark was designed against.
EXPECTED = {"companies": 1171373, "financial_data": 1170290}

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]

print("MongoDB", db.command("buildInfo")["version"])
print()
for name in ["companies", "financial_data"]:
    n = db[name].count_documents({})
    st = db.command("collstats", name)
    print("%-16s %9d docs  (expected %d, %s)"
          % (name, n, EXPECTED[name], "unchanged" if n == EXPECTED[name] else "CHANGED"))
    print("%-16s %9.2f GB on disk, avg doc %d bytes, %d indexes"
          % ("", st["size"] / 1024**3, st["avgObjSize"], st["nindexes"]))

MongoDB 8.3.8

companies          1171373 docs  (expected 1171373, unchanged)
                      1.57 GB on disk, avg doc 1439 bytes, 3 indexes
financial_data     1170290 docs  (expected 1170290, unchanged)
                      0.67 GB on disk, avg doc 611 bytes, 1 indexes


## Indexes

W1 joins on `organisasjonsnummer`. If that index is missing from
`companies`,the `$lookup` degrades to a collection scan per probe and the slow
down is fullyexplained.

In [2]:
for name in ["companies", "financial_data"]:
    print("%s:" % name)
    for idx, spec in db[name].index_information().items():
        print("   %-40s %s" % (idx, spec.get("key")))

companies:
   _id_                                     [('_id', 1)]
   organisasjonsnummer_1                    [('organisasjonsnummer', 1)]
   organisasjonsform.kode_1                 [('organisasjonsform.kode', 1)]
financial_data:
   _id_                                     [('_id', 1)]


## Which cache counters does this build expose?

WiredTiger statistic names change between MongoDB versions. An earlier version
of this notebook hardcoded `pages evicted by application threads `, which does
not exist in 8.3.8 and raised a `KeyError`. The names are therefore discovered
here rather than assumed.

Run this cell and read the output before the next one.

In [3]:
wt_cache = db.command("serverStatus")["wiredTiger"]["cache"]

print("cache statistics mentioning eviction or cache reads:")
for k in sorted(wt_cache):
    if "evict" in k.lower() or "read into" in k.lower():
        print("   %-72s %s" % (k, wt_cache[k]))

print("\ntotal cache statistics available on this build: %d" % len(wt_cache))

cache statistics mentioning eviction or cache reads:
   application requested eviction interrupt                                 0
   application thread time evicting (usecs)                                 0
   application threads eviction requested with cache fill ratio < 25%       0
   application threads eviction requested with cache fill ratio >= 25% and < 50% 0
   application threads eviction requested with cache fill ratio >= 50% and < 75% 0
   application threads eviction requested with cache fill ratio >= 75%      0
   application threads eviction skip page with updates or dirty page        0
   bytes read into cache                                                    2471836775
   checkpoint blocked page eviction                                         0
   checkpoint of history store file blocked non-history store page eviction 0
   dirty internal page cannot be evicted in disaggregated storage           0
   evict page attempts by eviction server                             

## Cache state

`bytes read into cache ` rising sharply during a run means data is being pulled
from disk rather than served from the WiredTiger cache, which is the
fingerprint of eviction by the Spark reads.

That counter is the one the contention hypothesis actually turns on. An
eviction count is read as corroboration where the build provides one.
Eachstatistic is looked up through a list of candidate names and reported as
unavailable if none match, so a renamed counter degrades the diagnostic instead
of aborting it.

In [4]:
def cache_stats():
    """WiredTiger cache counters, tolerant of version-dependent key names."""
    wt = db.command("serverStatus")["wiredTiger"]["cache"]

    def first(*names):
        for n in names:
            if n in wt:
                return wt[n]
        return None

    return {
        "in_cache_bytes": first("bytes currently in the cache"),
        "max_bytes": first("maximum bytes configured"),
        "read_into_cache_bytes": first("bytes read into cache"),
        "pages_evicted": first(
            "pages evicted by application threads",
            "application threads page read from disk to cache count",
            "pages evicted",
            "eviction worker thread evicting pages",
        ),
    }


def show_cache(label, st):
    print(label)
    for key, val in st.items():
        if val is None:
            print("   %-24s unavailable on this build" % key)
        elif key.endswith("_bytes"):
            print("   %-24s %.2f GB" % (key, val / 1024**3))
        else:
            print("   %-24s %d" % (key, val))


before = cache_stats()
show_cache("cache state before:", before)

cache state before:
   in_cache_bytes           2.54 GB
   max_bytes                7.10 GB
   read_into_cache_bytes    2.30 GB
   pages_evicted            29507


## The measurement

Same pipeline, with and without `allowDiskUse`. Each is warmed once and then
timed three times, matching the benchmark's method.

In [5]:
W1_PIPELINE = [
    {"$lookup": {"from": "companies", "localField": "organisasjonsnummer",
                 "foreignField": "organisasjonsnummer", "as": "company"}},
    {"$unwind": "$company"},
    {"$match": {"company.organisasjonsform.kode": "AS"}},
    {"$group": {"_id": "$fetch_status", "count": {"$sum": 1}}},
]

results = {}
for flag in [False, True]:
    list(db.financial_data.aggregate(W1_PIPELINE, allowDiskUse=flag))   # warm
    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        rows = list(db.financial_data.aggregate(W1_PIPELINE, allowDiskUse=flag))
        times.append(time.perf_counter() - t0)
    results[flag] = {"times": times,
                     "result": sorted((r["_id"], r["count"]) for r in rows)}
    print("allowDiskUse=%-5s median %6.2fs   min %6.2fs   max %6.2fs   %s"
          % (flag, statistics.median(times), min(times), max(times),
             results[flag]["result"]))

allowDiskUse=False median  42.01s   min  41.80s   max  43.35s   [('no_data', 27670), ('success', 403782)]
allowDiskUse=True  median  41.67s   min  41.47s   max  41.75s   [('no_data', 27670), ('success', 403782)]


In [6]:
after = cache_stats()
show_cache("cache state after:", after)


def delta(key):
    a, b = after.get(key), before.get(key)
    return None if a is None or b is None else a - b


d_read = delta("read_into_cache_bytes")
d_evict = delta("pages_evicted")

print("\nread into cache during this notebook: %s"
      % ("unavailable" if d_read is None else "%.2f GB" % (d_read / 1024**3)))
print("pages evicted during this notebook:   %s"
      % ("unavailable" if d_evict is None else str(d_evict)))

cache state after:
   in_cache_bytes           2.54 GB
   max_bytes                7.10 GB
   read_into_cache_bytes    2.30 GB
   pages_evicted            29507

read into cache during this notebook: 0.00 GB
pages evicted during this notebook:   0


## Query plan

`executionStats` shows whether the `$lookup` uses the `organisasjonsnummer`
index (`IXSCAN`) or falls back to a collection scan (`COLLSCAN`), and how many
documents were examined to produce the result.

In [7]:
exp = db.command({"explain": {"aggregate": "financial_data", "pipeline": W1_PIPELINE,
                              "cursor": {}},
                  "verbosity": "executionStats"})


def find_keys(obj, keys, path=""):
    """Pull selected fields out of the plan tree without printing all of it."""
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = path + "." + k if path else k
            if k in keys and not isinstance(v, (dict, list)):
                out.append((p, v))
            out += find_keys(v, keys, p)
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            out += find_keys(v, keys, "%s[%d]" % (path, i))
    return out


INTERESTING = {"stage", "indexName", "nReturned", "totalDocsExamined",
               "totalKeysExamined", "executionTimeMillis", "indexesUsed",
               "collectionScans"}
found = find_keys(exp, INTERESTING)
if not found:
    # Plan shape varies by version; fall back to dumping the top of the tree.
    print("No expected keys found. Top of plan:")
    print(json.dumps(exp, indent=1, default=str)[:3000])
else:
    for p, v in found:
        print("%-64s %s" % (p, v))

stages[0].$cursor.queryPlanner.winningPlan.stage                 PROJECTION_SIMPLE
stages[0].$cursor.queryPlanner.winningPlan.inputStage.stage      COLLSCAN
stages[0].$cursor.executionStats.nReturned                       1170290
stages[0].$cursor.executionStats.executionTimeMillis             48302
stages[0].$cursor.executionStats.totalKeysExamined               0
stages[0].$cursor.executionStats.totalDocsExamined               1170290
stages[0].$cursor.executionStats.executionStages.stage           PROJECTION_SIMPLE
stages[0].$cursor.executionStats.executionStages.nReturned       1170290
stages[0].$cursor.executionStats.executionStages.inputStage.stage COLLSCAN
stages[0].$cursor.executionStats.executionStages.inputStage.nReturned 1170290
stages[0].nReturned                                              1170290
stages[1].nReturned                                              431452
stages[1].totalDocsExamined                                      1170291
stages[1].totalKeysExamined     

## SaveWritten to disk so the figures can be quoted in the report without re-running.

In [8]:
out = {
    "mongodb_version": db.command("buildInfo")["version"],
    "collection_counts": {n: db[n].count_documents({})
                          for n in ["companies", "financial_data"]},
    "expected_counts": EXPECTED,
    "indexes": {n: {i: s.get("key") for i, s in db[n].index_information().items()}
                for n in ["companies", "financial_data"]},
    "w1_timings_seconds": {str(k): v["times"] for k, v in results.items()},
    "w1_result": {str(k): v["result"] for k, v in results.items()},
    "cache_before": before,
    "cache_after": after,
    "cache_delta": {"read_into_cache_bytes": d_read, "pages_evicted": d_evict},
    "earlier_benchmark_w1_seconds": 7.13,
    "current_benchmark_w1_seconds": 45.97,
}

with open("/home/jovyan/data/diagnose_variant_a.json", "w") as fh:
    json.dump(out, fh, indent=2, default=str)
print("Saved to data/diagnose_variant_a.json")

Saved to data/diagnose_variant_a.json


## Interpretation

Read the output against the table at the top:- **A is fast here (near 7s) and
slow in the benchmark** — the cause is cross-variant contention. The reordered
benchmark should fix it; if it does not, run each variant in its own kernel via
`SELECTED_VARIANTS`.- **`allowDiskUse=True` is materially slower than `False`**
— the flag is the cause. It was added for W4, which needs it; W1 does not, so
set it per workload rather than uniformly.- **A is slow here too, and both
flags agree** — the cause is the data or the plan. Check the counts against the
snapshot and look for `COLLSCAN` or a large `totalDocsExamined` in the plan
output.- **`read into cache ` is large during this notebook** — data was being
fetched from disk, which points back to eviction.

Whatever the outcome, record it in the report. A 6.4x unexplained variance in a
benchmark figure is a finding in its own right, and the honest thing to do with
the earlier 7.13s number is to state the conditions under which each was
measured rather than quietly keeping the more flattering one.